# 01 — Test Redis (cache + pub/sub)

Smoke test du container `fxvol-redis`. Aucune dépendance — Redis est une **feuille** dans le graphe (cf. `docs/container_deps.md`).

**Couvre** :
1. Container UP + `INFO` server (version, role)
2. Cache pur : `SET`/`GET`/`DEL` + `TTL` sur les vrais patterns de clés (`latest_spot:*`, `latest_vol_surface:*`, `heartbeat:*`)
3. Pub/Sub : 1 subscriber + 1 publisher + roundtrip sur les 5 channels du bus (`ticks`, `vol_update`, `risk_update`, `account`, `system_alerts`)
4. Memory : `maxmemory` policy + usage actuel
5. Hardening : commandes dangereuses (`FLUSHDB`, `FLUSHALL`, `CONFIG`) doivent être renamed/désactivées par `infrastructure/redis/redis.conf`

**Préreq** :
- Stack démarrée (au moins `redis` healthy) : `.\scripts\start_stack.ps1`
- Pas de secret nécessaire — Redis local sans password (acceptable car réseau Docker interne, pas exposé en prod)
- `pip install redis` (déjà dans requirements.txt)

**Référence** : `infrastructure/redis/redis.conf` + `src/bus/keys.py` (templates de keys + TTL).

## Setup — connexion + helpers

In [1]:
import subprocess
import time
from datetime import datetime, timezone

import redis

REDIS_URL = "redis://localhost:6380/0"
results = []

r = redis.from_url(REDIS_URL, decode_responses=True)
pong = r.ping()
if not pong:
    raise RuntimeError(
        "Redis baseline KO. Container down ?\n"
        "  -> .\\scripts\\start_stack.ps1 ; vérifie 'docker compose ps' = redis healthy"
    )
print(f"Connected -> {REDIS_URL}")
print(f"PING reply: {pong}")

def record(label, ok, detail=""):
    results.append((label, ok, detail))
    sym = "OK" if ok else "FAIL"
    print(f"  [{sym:4}] {label}{('  | ' + detail) if detail else ''}")

# Marqueur de session pour identifier les keys créées par CE notebook (cleanup à la fin)
MARKER = f"_test_{int(time.time())}"

Connected -> redis://localhost:6380/0
PING reply: True


## 1. Container + INFO server

**Ce que tu dois voir** : container `running`, version Redis ≥ 7, role `master` (pas de cluster en dev), uptime > 0.

In [2]:
print("== container + INFO ==")

# Container running ?
out = subprocess.run(
    ["docker", "inspect", "-f", "{{.State.Status}}", "fxvol-redis"],
    capture_output=True, text=True,
)
state = out.stdout.strip()
record("docker container state", state == "running", state)

# INFO server section
info = r.info("server")
ver = info.get("redis_version", "?")
uptime = info.get("uptime_in_seconds", 0)
record("INFO server", ver.startswith("7"), f"version={ver}, uptime={uptime}s")

# INFO replication section (role)
repl = r.info("replication")
role = repl.get("role", "?")
record("role", role == "master", f"role={role}")

# DBSIZE — nombre de clés actuel (peut être >0 si engines tournent)
dbsize = r.dbsize()
print(f"  [INFO] DBSIZE = {dbsize} keys")

== container + INFO ==
  [OK  ] docker container state  | running
  [OK  ] INFO server  | version=7.4.8, uptime=6253s
  [OK  ] role  | role=master
  [INFO] DBSIZE = 1 keys


## 2. Cache pur — SET / GET / DEL + TTL

**Ce que tu dois voir** : tous les patterns de clés du `src/bus/keys.py` fonctionnent (`latest_spot:*`, `latest_vol_surface:*`, `heartbeat:*`, etc.). TTL bien appliqué.

In [3]:
print("== cache SET/GET/DEL + TTL ==")

# Cas 1 : latest_spot avec TTL 30s (valeur scalaire simple)
key = f"latest_spot:EURUSD{MARKER}"
r.set(key, "1.08234", ex=30)
v = r.get(key)
ttl = r.ttl(key)
record("SET/GET latest_spot avec TTL", v == "1.08234" and 0 < ttl <= 30, f"value={v}, ttl={ttl}s")

# Cas 2 : latest_vol_surface avec TTL 600s (valeur JSONB sérialisée)
import json
key = f"latest_vol_surface:EURUSD{MARKER}"
payload = {"spot": 1.08234, "strikes": [1.05, 1.08, 1.11], "vols": [0.085, 0.078, 0.083]}
r.set(key, json.dumps(payload), ex=600)
got = json.loads(r.get(key))
record("SET/GET latest_vol_surface JSON roundtrip", got == payload, f"keys={list(got.keys())}")

# Cas 3 : heartbeat:<engine> — pattern utilisé pour monitor les engines
for engine in ("market_data", "vol_engine", "risk_engine", "db_writer"):
    key = f"heartbeat:{engine}{MARKER}"
    r.set(key, datetime.now(timezone.utc).isoformat(), ex=30)
record("SET 4 heartbeat keys avec TTL 30s", True, "market_data, vol_engine, risk_engine, db_writer")

# Cas 4 : KEYS pattern — récupère toutes nos test keys
matched = r.keys(f"*{MARKER}")
record(f"KEYS *{MARKER}", len(matched) >= 6, f"found {len(matched)} keys")

# Cas 5 : DEL multi-keys
if matched:
    deleted = r.delete(*matched)
    record("DEL multi-keys", deleted == len(matched), f"deleted {deleted}/{len(matched)}")

# Vérif : les keys sont bien parties
still_there = r.keys(f"*{MARKER}")
record("verify cleanup", len(still_there) == 0, f"remaining: {still_there}")

== cache SET/GET/DEL + TTL ==
  [OK  ] SET/GET latest_spot avec TTL  | value=1.08234, ttl=30s
  [OK  ] SET/GET latest_vol_surface JSON roundtrip  | keys=['spot', 'strikes', 'vols']
  [OK  ] SET 4 heartbeat keys avec TTL 30s  | market_data, vol_engine, risk_engine, db_writer
  [OK  ] KEYS *_test_1777302366  | found 6 keys
  [OK  ] DEL multi-keys  | deleted 6/6
  [OK  ] verify cleanup  | remaining: []


## 3. Pub/Sub — roundtrip sur les 5 channels du bus

Channels du bus définis dans `src/bus/channels.py` :

- `ticks` (publié par `market-data`)
- `vol_update` (publié par `vol-engine`)
- `risk_update` (publié par `risk-engine`)
- `account` (publié par `market-data`)
- `system_alerts` (publié par n'importe quel service en cas d'anomalie)

Pour chaque channel : un subscriber dans un thread + un publisher qui envoie 1 message + on vérifie qu'il arrive en <1s. C'est exactement le mécanisme que le `api/ws.py` utilise pour bridger Redis → WebSocket vers le frontend.

In [4]:
print("== pub/sub roundtrip ==")

CHANNELS = ("ticks", "vol_update", "risk_update", "account", "system_alerts")

for ch in CHANNELS:
    sub = redis.from_url(REDIS_URL, decode_responses=True).pubsub(ignore_subscribe_messages=True)
    sub.subscribe(ch)

    # Drain l'éventuel ack de subscribe
    time.sleep(0.05)

    # Publish
    test_msg = json.dumps({"channel": ch, "marker": MARKER, "ts": datetime.now(timezone.utc).isoformat()})
    n_received = r.publish(ch, test_msg)

    # Wait + read
    deadline = time.time() + 1.0
    received = None
    while time.time() < deadline:
        msg = sub.get_message(timeout=0.1)
        if msg and msg.get("type") == "message":
            received = msg["data"]
            break

    sub.unsubscribe(ch)
    sub.close()

    ok = received is not None and json.loads(received).get("marker") == MARKER
    record(f"PUB/SUB {ch}", ok, f"n_subscribers={n_received}, received={'OK' if ok else 'NONE'}")

== pub/sub roundtrip ==
  [OK  ] PUB/SUB ticks  | n_subscribers=2, received=OK
  [OK  ] PUB/SUB vol_update  | n_subscribers=2, received=OK
  [OK  ] PUB/SUB risk_update  | n_subscribers=2, received=OK
  [OK  ] PUB/SUB account  | n_subscribers=2, received=OK
  [OK  ] PUB/SUB system_alerts  | n_subscribers=2, received=OK


## 4. Memory + maxmemory policy

**Ce que tu dois voir** : `maxmemory` configurée à 256mb (cf. `infrastructure/redis/redis.conf`) avec policy `allkeys-lru` (éviction des moins-récemment-utilisés). Mémoire actuelle bien sous le cap.

Si `maxmemory` = 0 → policy par défaut `noeviction` → un overflow de cache crashe les SET avec `OOM command not allowed`. Mauvais réflexe.

In [5]:
print("== memory + maxmemory ==")

mem = r.info("memory")
used = mem.get("used_memory_human", "?")
rss = mem.get("used_memory_rss_human", "?")
maxmem = mem.get("maxmemory_human", "?")
policy = mem.get("maxmemory_policy", "?")
print(f"  used_memory:        {used}")
print(f"  used_memory_rss:    {rss}")
print(f"  maxmemory:          {maxmem}")
print(f"  maxmemory_policy:   {policy}")

record("maxmemory bornée (≠ 0)", mem.get("maxmemory", 0) > 0, f"maxmemory={maxmem}")
record("policy = allkeys-lru", policy == "allkeys-lru", f"policy={policy}")

== memory + maxmemory ==
  used_memory:        1.22M
  used_memory_rss:    8.75M
  maxmemory:          256.00M
  maxmemory_policy:   allkeys-lru
  [OK  ] maxmemory bornée (≠ 0)  | maxmemory=256.00M
  [OK  ] policy = allkeys-lru  | policy=allkeys-lru


## 5. Hardening — commandes dangereuses désactivées

**Ce que tu dois voir** : `FLUSHDB`, `FLUSHALL`, `CONFIG` retournent `unknown command` (renamed à vide dans `infrastructure/redis/redis.conf`). Defense in depth : un attaquant qui aurait accès au socket Redis ne peut pas wiper la cache ni reconfigurer le serveur.

Si l'une de ces commandes répond normalement, le hardening config n'est pas appliqué — corriger `redis.conf` + restart.

In [6]:
print("== commandes dangereuses désactivées ==")

for cmd in ("FLUSHDB", "FLUSHALL", "CONFIG"):
    try:
        # On exécute la commande brute. Doit lever une ResponseError si renamed.
        r.execute_command(cmd)
        record(f"{cmd} bloquée", False, f"⚠ {cmd} a fonctionné — hardening manquant !")
    except redis.exceptions.ResponseError as e:
        msg = str(e).lower()
        blocked = "unknown command" in msg or "err" in msg
        record(f"{cmd} bloquée", blocked, f"got: {e}")
    except Exception as e:
        record(f"{cmd} bloquée", False, f"unexpected exception: {type(e).__name__}: {e}")

== commandes dangereuses désactivées ==
  [OK  ] FLUSHDB bloquée  | got: unknown command 'FLUSHDB', with args beginning with: 
  [OK  ] FLUSHALL bloquée  | got: unknown command 'FLUSHALL', with args beginning with: 
  [OK  ] CONFIG bloquée  | got: unknown command 'CONFIG', with args beginning with: 


## Récap final

Tableau OK/FAIL + cleanup final.

In [7]:
# Cleanup défensif : supprime toute key qui contiendrait le marker (au cas où)
leftover = r.keys(f"*{MARKER}")
if leftover:
    r.delete(*leftover)
    print(f"Cleanup: deleted {len(leftover)} leftover keys")

n_ok = sum(1 for _, ok, _ in results if ok)
n_fail = sum(1 for _, ok, _ in results if not ok)

print(f"\n{'LABEL':<45} STATUS  DETAIL")
print("-" * 100)
for label, ok, detail in results:
    sym = "OK" if ok else "FAIL"
    print(f"{label:<45} {sym:<6}  {detail}")
print("-" * 100)
print(f"\n{n_ok} OK / {n_fail} FAIL  ({len(results)} total)")

if n_fail:
    print("\n⚠ FAILs detected. Common causes:")
    print("  - container redis pas démarré         -> docker compose ps")
    print("  - port 6380 pas exposé sur l'host      -> docker-compose.override.yml")
    print("  - hardening commands manquant          -> infrastructure/redis/redis.conf")
    print("                                            (rename-command FLUSHDB/FLUSHALL/CONFIG)")
    print("  - maxmemory policy != allkeys-lru      -> idem, ligne 'maxmemory-policy'")
else:
    print("\n✅ redis container fully validated (cache + pub/sub + hardening).")


LABEL                                         STATUS  DETAIL
----------------------------------------------------------------------------------------------------
docker container state                        OK      running
INFO server                                   OK      version=7.4.8, uptime=6253s
role                                          OK      role=master
SET/GET latest_spot avec TTL                  OK      value=1.08234, ttl=30s
SET/GET latest_vol_surface JSON roundtrip     OK      keys=['spot', 'strikes', 'vols']
SET 4 heartbeat keys avec TTL 30s             OK      market_data, vol_engine, risk_engine, db_writer
KEYS *_test_1777302366                        OK      found 6 keys
DEL multi-keys                                OK      deleted 6/6
verify cleanup                                OK      remaining: []
PUB/SUB ticks                                 OK      n_subscribers=2, received=OK
PUB/SUB vol_update                            OK      n_subscribers=2, receiv